<a href="https://colab.research.google.com/github/Dara0510/BorisovaDaria/blob/main/%D0%B1%D0%BB%D0%BE%D0%BA%D0%BD%D0%BE%D1%82_%22hw_clustering_ipynb%22.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🕵️‍♂️ Домашнее задание: Детективное агентство "Кластеризация"

**Легенда:** Мы перехватили архив с 5000 загадочных изображений (`lyubachuba/mystery_images_5k`). У нас нет никаких подписей и категорий.

**Ваша задача:** использовать алгоритмы машинного обучения (без учителя), чтобы найти скрытую структуру в этих данных, сгруппировать похожие картинки и сделать выводы.

Мы используем легкую нейросеть (ResNet-18), чтобы быстро векторизовать картинки на GPU. Ваша цель — применить методы кластеризации, **подобрать параметры** и навайбкодить (с помощью ИИ или самостоятельно) **красивые визуализации** для каждого метода.

In [ ]:
!pip install -q datasets torch torchvision umap-learn plotly seaborn scikit-learn hdbscan pandas

import torch
import torchvision.transforms as T
import torchvision.models as models
from datasets import load_dataset
import numpy as np
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

import warnings
warnings.filterwarnings('ignore')

### 1. Загрузка данных и Векторизация (Feature Extraction)
Используем ResNet-18. Эта сеть вычисляет векторы размером 512 чисел для каждой картинки. Она работает очень быстро и дает отличный баланс скорости и качества.


### ⚠️ ВАЖНО: Включаем GPU (Видеокарту)!

Нейросети (даже такие быстрые, как ResNet-18) работают очень медленно на обычном процессоре (CPU). Если оставить всё как есть, векторизация займет 20-30 минут. На видеокарте (GPU) это займет меньше 1 минуты.

**Как включить GPU в Google Colab:**
1. В верхнем меню нажмите: **Среда выполнения** (Runtime).
2. Выберите: **Сменить среду выполнения** (Change runtime type).
3. В разделе "Аппаратный ускоритель" (Hardware accelerator) выберите **T4 GPU** (или любой другой доступный GPU).
4. Нажмите **Сохранить**.

*Проверка:* При запуске ячейки ниже должно напечататься `Используем устройство: cuda`. Если написано `cpu` — вы забыли включить GPU!

In [ ]:
print("Загрузка датасета...")
dataset = load_dataset("lyubachuba/mystery_images_5k", split="train")

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Используем устройство: {device}")

# Загружаем ResNet-18 без последнего слоя классификации
resnet = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
resnet.fc = torch.nn.Identity() # Убираем слой классификации, оставляем эмбеддинги (512)
resnet = resnet.to(device).eval()

# Предобработка картинок для ResNet
transform = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

def extract_features(batch):
    images = [transform(img.convert("RGB")) for img in batch["image"]]
    images_tensor = torch.stack(images).to(device)
    with torch.no_grad():
        embeddings = resnet(images_tensor).cpu().numpy()
    return {"embedding": embeddings}

print("Извлекаем признаки (векторизация)...")

dataset_emb = dataset.map(
    extract_features,
    batched=True,
    batch_size=64,
    desc="Векторизация (ResNet-18)"
)

X = np.array(dataset_emb['embedding'])
print(f"Готово! Размерность данных X: {X.shape}")

### 2. Снижение размерности (UMAP)
Алгоритмам плотности и нам для визуализации нужны данные меньшей размерности.

In [ ]:
import umap

# Сжимаем 512 -> 3 компоненты для 3D графиков и лучшей кластеризации плотностью
reducer = umap.UMAP(n_components=3, random_state=42)
X_umap = reducer.fit_transform(X)
print(f"Размерность X_umap: {X_umap.shape}")

---
## 🚀 Ваша часть задания (Сделайте выводы в текстовых ячейках под кодом!)

### Задание 1. K-Means
**Что нужно:**
1. Попробуйте разное количество кластеров (`n_clusters`). Посчитайте метрики (Silhouette, Davies-Bouldin). Выберите количество кластеров через метод локтя.
2. **Визуализация (Vibe-code):** Напишите код для интерактивного 3D Scatter plot через `plotly.express` по данным `X_umap`, где точки раскрашены в цвета кластеров.
3. Выведите сетку 3x3 случайных картинок для 2-3 интересных кластеров.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score
import plotly.express as px
import matplotlib.pyplot as plt
import numpy as np

inertias = []
silhouette_scores = []
db_scores = []
K_range = range(3, 11)

for k in K_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init='auto')
    labels = kmeans.fit_predict(X_umap)

    inertias.append(kmeans.inertia_)
    silhouette_scores.append(silhouette_score(X_umap, labels))
    db_scores.append(davies_bouldin_score(X_umap, labels))
    print(f"K={k}: Inertia={kmeans.inertia_:.2f}, Silhouette={silhouette_scores[-1]:.4f}, Davies-Bouldin={db_scores[-1]:.4f}")

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].plot(K_range, inertias, 'bo-')
axes[0].set_title( "Inertia")
axes[0].set_xlabel('Количество кластеров K')
axes[0].grid(True)

axes[1].plot(K_range, silhouette_scores, 'ro-')
axes[1].set_title('Silhouette Score')
axes[1].set_xlabel('Количество кластеров K')
axes[1].grid(True)

axes[2].plot(K_range, db_scores, 'go-')
axes[2].set_title('Davies-Bouldin Score')
axes[2].set_xlabel('Количество кластеров K')
axes[2].grid(True)

plt.tight_layout()
plt.show()

optimal_k = K_range[np.argmax(silhouette_scores)]
print(f"\nОптимальное количество кластеров по Silhouette Score: {optimal_k}")

final_kmeans = KMeans(n_clusters=optimal_k, random_state=42, n_init='auto')
final_labels = final_kmeans.fit_predict(X_umap)

import pandas as pd
plot_df = pd.DataFrame(X_umap, columns=['UMAP1', 'UMAP2', 'UMAP3'])
plot_df['Cluster'] = final_labels.astype(str)

fig = px.scatter_3d(plot_df, x='UMAP1', y='UMAP2', z='UMAP3', color='Cluster',
                    title=f'K-Means с K={optimal_k}',
                    color_discrete_sequence=px.colors.qualitative.Vivid)
fig.show()

def show_image_grid(images, title="Кластер"):
    fig, axes = plt.subplots(3, 3, figsize=(8, 8))
    fig.suptitle(title, fontsize=16)
    for i, ax in enumerate(axes.flat):
        if i < len(images):
            ax.imshow(images[i])
        ax.axis('off')
    plt.tight_layout()
    plt.show()

def get_images_from_cluster(labels, cluster_num, dataset, max_images=9):
    indices = np.where(labels == cluster_num)[0]
    if len(indices) == 0:
        return []
    selected_indices = np.random.choice(indices, size=min(max_images, len(indices)), replace=False)
    return [dataset[int(i)]['image'] for i in selected_indices]

interesting_clusters = [0, 1, 2]
for cluster_num in interesting_clusters:
    if cluster_num < optimal_k:
        cluster_images = get_images_from_cluster(final_labels, cluster_num, dataset_emb)
        show_image_grid(cluster_images, title=f"Кластер {cluster_num}")


**Выводы по K-Means:**
*(Напишите здесь: Легко ли алгоритм нашел кластеры? Какое количество кластеров кажется оптимальным? Что общего у картинок внутри кластеров?)*

Алгоритм K-Means хорошо справился — оптимальным оказалось 3 кластера (Silhouette 0.5053, Davies-Bouldin 0.7348). При увеличении K метрики шли только вниз, так что три группы — самый логичный вариант.

Я вижу, что кластеры разделились так: в первом - пейзажи, во втором - портреты, а в третьем  изображения с другим стилем рисовки. Алгоритм уловил  что-то более содержательное, чем цвета

### Задание 2. Иерархическая кластеризация
**Что нужно:**
1. Попробуйте параметры `linkage` ('ward', 'average', 'complete').
2. **Визуализация (Vibe-code):** Постройте раскрашенную дендрограмму (используйте `scipy.cluster.hierarchy.dendrogram`), взяв случайную подвыборку в 150-200 картинок (иначе график превратится в кашу).
3. Постройте 2D Scatter plot (Seaborn), визуально сравнив кластеры от разных `linkage`.

In [ ]:
from sklearn.cluster import AgglomerativeClustering
from scipy.cluster.hierarchy import dendrogram, linkage
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

np.random.seed(42)
sample_indices = np.random.choice(X_umap.shape[0], size=200, replace=False)
X_sample = X_umap[sample_indices]

linkage_methods = ['ward', 'average', 'complete']
n_clusters = 3

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for i, method in enumerate(linkage_methods):
    Z = linkage(X_sample, method=method)
    dendrogram(Z, ax=axes[i], labels=np.arange(200), leaf_rotation=90, leaf_font_size=8)
    axes[i].set_title(f'Дендрограмма (linkage={method})')
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for i, method in enumerate(linkage_methods):
    clustering = AgglomerativeClustering(n_clusters=n_clusters, linkage=method)
    labels_sample = clustering.fit_predict(X_sample)

    sns.scatterplot(x=X_sample[:, 0], y=X_sample[:, 1], hue=labels_sample, palette='viridis', ax=axes[i], legend='full')
    axes[i].set_title(f'Кластеры (linkage={method})')
plt.tight_layout()
plt.show()


**Выводы по Иерархической кластеризации:**
*(Напишите здесь: Какой метод linkage дал самые осмысленные формы кластеров на графике? Видно ли по дендрограмме, на сколько групп логично бить датасет? Как выглядит иерархия?)*

 ward и complete дают почти одинаковую картину на кластерах, но при этом дендрограмма ward - лучше. Лучший вариант - 3 кластера. Иерархия выглядит так: сначала отделяются крупные содержательные блоки, а уже внутри них можно искать более мелкие подгруппы, если нужно.

### Задание 3. DBSCAN
**Что нужно:**
1. Подберите `eps` и `min_samples` в цикле (попробуйте разные комбинации). Посчитайте % шума для каждой.
2. **Визуализация (Vibe-code):** Сделайте график (Matplotlib), показывающий точки-кластеры в цвете, а точки-шум серым цветом (`label == -1`).
3. Выведите галерею "выбросов" (самых странных картинок, которые алгоритм отнес к шуму). Выведите то, что принадлежит к одному кластеру

In [ ]:
from sklearn.cluster import DBSCAN
import matplotlib.pyplot as plt
import numpy as np

eps_values = [0.3, 0.5, 0.7, 0.9]
min_samples_values = [5, 10, 15]

print("Подбор параметров для DBSCAN:")
best_labels = None
best_params = {'eps': 0.5, 'min_samples': 10}
best_clusters = 0
best_noise = 100

for eps_val in eps_values:
    for min_samp in min_samples_values:
        db = DBSCAN(eps=eps_val, min_samples=min_samp)
        labels = db.fit_predict(X_umap)

        n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
        n_noise = list(labels).count(-1)
        noise_percent = 100 * n_noise / len(labels)

        print(f"eps={eps_val}, min_samples={min_samp} -> Кластеров: {n_clusters}, Шума: {noise_percent:.2f}%")

        if n_clusters >= 2 and noise_percent < 30:
            if n_clusters > best_clusters or (n_clusters == best_clusters and noise_percent < best_noise):
                best_clusters = n_clusters
                best_noise = noise_percent
                best_params = {'eps': eps_val, 'min_samples': min_samp}
                best_labels = labels.copy()

if best_labels is None:
    best_labels = DBSCAN(eps=0.5, min_samples=10).fit_predict(X_umap)
    best_params = {'eps': 0.5, 'min_samples': 10}

print(f"\nЛучшие параметры: eps={best_params['eps']}, min_samples={best_params['min_samples']}")
print(f"Кластеров: {len(set(best_labels)) - (1 if -1 in best_labels else 0)}, Шума: {100 * list(best_labels).count(-1) / len(best_labels):.2f}%")

plt.figure(figsize=(10, 6))
unique_labels = set(best_labels)
colors = plt.cm.Spectral(np.linspace(0, 1, len(unique_labels)))

for k, col in zip(unique_labels, colors):
    if k == -1:
        col = [0.7, 0.7, 0.7, 0.5]
    class_member_mask = (best_labels == k)
    xy = X_umap[class_member_mask]
    plt.plot(xy[:, 0], xy[:, 1], 'o', markerfacecolor=tuple(col), markeredgecolor='k', markersize=5, alpha=0.6)

plt.title(f'DBSCAN: eps={best_params["eps"]}, min_samples={best_params["min_samples"]}')
plt.show()

def show_image_grid(images, title=""):
    fig, axes = plt.subplots(3, 3, figsize=(8, 8))
    fig.suptitle(title, fontsize=16)
    for i, ax in enumerate(axes.flat):
        if i < len(images):
            ax.imshow(images[i])
        ax.axis('off')
    plt.tight_layout()
    plt.show()

noise_indices = np.where(best_labels == -1)[0]
sample_noise = np.random.choice(noise_indices, size=min(9, len(noise_indices)), replace=False)
noise_images = [dataset_emb[int(i)]['image'] for i in sample_noise]
show_image_grid(noise_images, title="Выбросы (шум)")

cluster_labels = set(best_labels) - {-1}
if len(cluster_labels) > 0:
    first_cluster = list(cluster_labels)[0]
    cluster_indices = np.where(best_labels == first_cluster)[0]
    sample_cluster = np.random.choice(cluster_indices, size=min(9, len(cluster_indices)), replace=False)
    cluster_images = [dataset_emb[int(i)]['image'] for i in sample_cluster]
    show_image_grid(cluster_images, title=f"Кластер {first_cluster}")

**Выводы по DBSCAN:**
*(Напишите здесь: Насколько сложно было подобрать параметры? Почему картинки-выбросы оказались в шуме?)*

Параметры капризны, адекватного варианта найти так и не удалось, некоторые нужные картинки улетют в шум

### Задание 4. HDBSCAN
**Что нужно:**
1. Примените алгоритм (попробуйте изменить `min_cluster_size`).
2. У HDBSCAN есть фича — уверенность в кластеризации (probabilities_).
3. **Визуализация (Vibe-code):** Нарисуйте 2D или 3D Scatter (Plotly), где прозрачность (`opacity`) или размер точки (`size`) зависит от того, насколько HDBSCAN уверен, что точка принадлежит кластеру.

In [ ]:
import hdbscan
import plotly.express as px
import pandas as pd

print("Запуск HDBSCAN...")
hdb = hdbscan.HDBSCAN(min_cluster_size=50, gen_min_span_tree=True)
hdb_labels = hdb.fit_predict(X_umap)

n_clusters = len(set(hdb_labels)) - (1 if -1 in hdb_labels else 0)
n_noise = list(hdb_labels).count(-1)
print(f"Кластеров: {n_clusters}, Шума: {n_noise} ({100*n_noise/len(hdb_labels):.2f}%)")

probabilities = hdb.probabilities_

size = 1 + 9 * (probabilities - probabilities.min()) / (probabilities.max() - probabilities.min())

plot_df = pd.DataFrame(X_umap, columns=['UMAP1', 'UMAP2', 'UMAP3'])
plot_df['Cluster'] = hdb_labels.astype(str)
plot_df['Probability'] = probabilities
plot_df['Size'] = size

fig = px.scatter_3d(plot_df, x='UMAP1', y='UMAP2', z='UMAP3',
                    color='Cluster',
                    size='Size',
                    opacity=0.7,
                    title='HDBSCAN: размер точки зависит от уверенности',
                    color_discrete_sequence=px.colors.qualitative.Vivid)
fig.show()

plt.figure(figsize=(10, 4))
plt.hist(probabilities[probabilities > 0.1], bins=50, alpha=0.7, color='blue')
plt.title('Распределение вероятностей принадлежности к кластеру')
plt.xlabel('Вероятность')
plt.ylabel('Количество точек')
plt.show()

**Выводы по HDBSCAN:**
*(Напишите здесь: Легче ли он настраивается, чем обычный DBSCAN? Много ли точек получили низкую вероятность?)*

HDBSCAN  легче. Достаточно задать минимальный размер кластера, и алгоритм сам разбирается, где плотно, а где пусто.

Алгоритм нашел 2 кластера и выкинул в шум почти 17% картинок. По гистограмме видно, что большинство точек получили высокую уверенность - это хорошо. Шума много, но главное, чтобы там картинки были похожи друг на друга



### 🏆 5. Итоговое расследование
Соберите все метрики (Silhouette, Davies-Bouldin, % шума) в единую `pandas` таблицу и раскрасьте её с помощью `DataFrame.style.background_gradient()`.

In [ ]:
import pandas as pd
from sklearn.metrics import silhouette_score, davies_bouldin_score

def get_metrics(labels, X, name):
    n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
    n_noise = list(labels).count(-1)
    noise_percent = 100 * n_noise / len(labels)

    if n_clusters > 1 and n_noise < len(labels):
        noise_mask = labels != -1
        sil = silhouette_score(X[noise_mask], labels[noise_mask])
        db = davies_bouldin_score(X[noise_mask], labels[noise_mask])
    else:
        sil = np.nan
        db = np.nan
    return {'Алгоритм': name, 'Кластеров': n_clusters, 'Шума %': noise_percent,
            'Silhouette': sil, 'Davies-Bouldin': db}

data = []
data.append(get_metrics(final_labels, X_umap, 'K-Means'))
data.append(get_metrics(best_labels, X_umap, 'DBSCAN'))
data.append(get_metrics(hdb_labels, X_umap, 'HDBSCAN'))

df = pd.DataFrame(data)
df.set_index('Алгоритм', inplace=True)

styled = df.style.background_gradient(cmap='RdYlGn', subset=['Silhouette', 'Davies-Bouldin'])
styled = styled.format({'Шума %': '{:.2f}%', 'Silhouette': '{:.4f}', 'Davies-Bouldin': '{:.4f}'})
display(styled)

### 🔍 Гранд-вывод:
1. Какой алгоритм оказался лучшим математически (по метрикам)?               K-Means
2. Какой алгоритм оказался лучшим визуально (субъективно)?  Ну для меня самый понятный linkage (ward)                                                                               
3. **Что это за датасет?** Посмотрев на картинки, какую общую тему/жанр вы бы им присвоили? Живопись! Мне кажется каких-то исторических времён
4. На какие группы уместнее всего разделить датасет? Ну вот почти как оно разделено в первом пункте: Пейзажи-портреты-что-то сюриалистическое как пикассо